## 追加したデータセットのクラスを本タスクにおけるクラスに変更

In [13]:
from pathlib import Path

# Roboflow側クラスID → SmartCloset側クラスID
#0 outer
#4 shoes
#5 bag
#6 hat
#7 watch
#8 glasses
CLASS_MAP = {
   # 0: 4,   # shoes
   # 1: 4,
   # 2: 4,
   0: 5,   # bag
   1: 5,
   2: 5,
   3: 5,
    # 0: 6,   # hat
    # 0: 7,   # watch
    # 1: 7,
    #0: 8,   # glasses
}

label_dirs = [
    "../dataset/fashion_reinforcement/bag/Bag Features.v4i.yolov8/train/labels",
    "../dataset/fashion_reinforcement/bag/Bag Features.v4i.yolov8/valid/labels",
    "../dataset/fashion_reinforcement/bag/Bag Features.v4i.yolov8/test/labels",
]

for label_dir in label_dirs:
    label_dir = Path(label_dir)

    if not label_dir.exists():
        print(f"{label_dir} が存在しません")
        continue

    for txt_file in label_dir.glob("*.txt"):
        new_lines = []

        with open(txt_file, "r") as f:
            for line in f:
                parts = line.strip().split()

                # 空行対策
                if len(parts) == 0:
                    continue

                old_id = int(parts[0])

                if old_id in CLASS_MAP:
                    parts[0] = str(CLASS_MAP[old_id])

                new_lines.append(" ".join(parts))

        with open(txt_file, "w") as f:
            f.write("\n".join(new_lines))

print("クラスID変換完了")

クラスID変換完了


## 全て移動したあとのyoloデータセットが問題ないかチェック

In [1]:
from pathlib import Path

for split in ["train", "val"]:
    img_dir = Path(f"../dataset/fashionpedia_yolo_9class/images/{split}")
    lbl_dir = Path(f"../dataset/fashionpedia_yolo_9class/labels/{split}")

    n_img = len(list(img_dir.glob("*")))
    n_lbl = len(list(lbl_dir.glob("*.txt")))

    print(split)
    print("images:", n_img)
    print("labels:", n_lbl)
    print("-" * 30)

train
images: 53370
labels: 53370
------------------------------
val
images: 1805
labels: 1805
------------------------------


In [2]:
classes = set()

for txt in Path("../dataset/fashionpedia_yolo_9class/labels").rglob("*.txt"):
    with open(txt) as f:
        for line in f:
            if line.strip():
                classes.add(int(line.split()[0]))

print(sorted(classes))

[0, 1, 2, 3, 4, 5, 6, 7, 8]


In [3]:
from collections import Counter
from pathlib import Path

counter = Counter()

for txt in Path("../dataset/fashionpedia_yolo_9class/labels").rglob("*.txt"):
    with open(txt) as f:
        for line in f:
            if line.strip():
                cls = int(line.split()[0])
                counter[cls] += 1

for cls in sorted(counter):
    print(cls, counter[cls])

0 24160
1 47021
2 27215
3 33837
4 56087
5 16788
6 8875
7 5425
8 6677


## outerクラスはobject detectionを追加してしまったので削除

In [2]:
from pathlib import Path

base = Path("../dataset/fashionpedia_yolo_9class")
outer = Path("../dataset/fashion_reinforcement/outer/zipper-detector.v1i.yolov8")  # 必要ならここを実際のouterフォルダ名に変更

for src_split, dst_split in [("train", "train"), ("valid", "train"), ("test", "val")]:
    src_img_dir = outer / src_split / "images"

    if not src_img_dir.exists():
        continue

    for img in src_img_dir.glob("*"):
        (base / "images" / dst_split / img.name).unlink(missing_ok=True)
        (base / "labels" / dst_split / f"{img.stem}.txt").unlink(missing_ok=True)

print("outer追加データを削除しました")

outer追加データを削除しました


In [3]:
from pathlib import Path

for split in ["train", "val"]:
    print(split)
    print("images:", len(list((Path("../dataset/fashionpedia_yolo_9class/images") / split).glob("*"))))
    print("labels:", len(list((Path("../dataset/fashionpedia_yolo_9class/labels") / split).glob("*.txt"))))

train
images: 52842
labels: 52842
val
images: 1792
labels: 1792


## 他のクラスにおいてもobject detectionが存在していたので犯人を探す

列挙

In [1]:
from pathlib import Path

for txt in Path("../dataset/fashionpedia_yolo_9class/labels").rglob("*.txt"):

    with open(txt) as f:
        for line in f:
            n = len(line.split())

            # detectionは5個
            if n == 5:
                print(txt)
                break

../dataset/fashionpedia_yolo_9class/labels/train/bag_00012_jpg.rf.3c3625e439de36c49982cd8acecc26ab.txt
../dataset/fashionpedia_yolo_9class/labels/train/bag_00522_jpg.rf.326348f12f461fcb30b7de15760bdab6.txt
../dataset/fashionpedia_yolo_9class/labels/train/bag_00101_jpg.rf.aadcdcc9be8f3816cb2258556317ff67.txt
../dataset/fashionpedia_yolo_9class/labels/train/bag_00750_jpg.rf.28ef7e44eee9ff8e13944b3440d7e824.txt
../dataset/fashionpedia_yolo_9class/labels/train/bag_00749_jpg.rf.7de3a78d15ea01271f104676c352803b.txt
../dataset/fashionpedia_yolo_9class/labels/train/bag_00893_jpg.rf.bcd3017ee8ee64507301bbd53104956d.txt
../dataset/fashionpedia_yolo_9class/labels/train/bag_00657_jpg.rf.0e66f0e8dc34040f6d45ceb09fd0c9b6.txt
../dataset/fashionpedia_yolo_9class/labels/train/bag_00134_jpg.rf.6625c2011c6fe3acc95bfcebb486434e.txt
../dataset/fashionpedia_yolo_9class/labels/train/bag_00490_jpg.rf.db18730c26ba8481300091f9c329eac9.txt
../dataset/fashionpedia_yolo_9class/labels/train/bag_00819_jpg.rf.6692a79

削除

In [2]:
from pathlib import Path

base = Path("../dataset/fashionpedia_yolo_9class")

image_exts = [".jpg", ".jpeg", ".png", ".webp"]

removed = 0

for txt in (base / "labels").rglob("*.txt"):
    has_detection_line = False

    with open(txt) as f:
        for line in f:
            if line.strip() and len(line.split()) == 5:
                has_detection_line = True
                break

    if has_detection_line:
        split = txt.parent.name  # train or val
        stem = txt.stem

        # label削除
        txt.unlink()
        removed += 1

        # 対応画像削除
        img_dir = base / "images" / split
        for ext in image_exts:
            img = img_dir / f"{stem}{ext}"
            if img.exists():
                img.unlink()

print("削除したBBox形式ラベル数:", removed)

削除したBBox形式ラベル数: 77


確認

In [3]:
from collections import Counter
from pathlib import Path

counter = Counter()

for txt in Path("../dataset/fashionpedia_yolo_9class/labels").rglob("*.txt"):
    with open(txt) as f:
        for line in f:
            if line.strip():
                counter[len(line.split())] += 1

print(counter)
print("5個の行数:", counter[5])

Counter({15: 5397, 13: 5352, 17: 5154, 11: 5086, 19: 4837, 21: 4672, 23: 4441, 25: 4317, 9: 4244, 27: 4124, 29: 3994, 33: 3897, 35: 3866, 31: 3862, 39: 3841, 43: 3740, 37: 3739, 41: 3679, 49: 3662, 47: 3605, 45: 3575, 51: 3461, 55: 3349, 53: 3276, 57: 3179, 59: 3134, 61: 2983, 63: 2889, 65: 2682, 67: 2535, 69: 2490, 71: 2474, 73: 2422, 79: 2201, 77: 2181, 75: 2156, 81: 1973, 85: 1952, 83: 1914, 87: 1830, 89: 1762, 91: 1689, 93: 1687, 97: 1603, 95: 1593, 99: 1505, 103: 1479, 101: 1476, 105: 1465, 7: 1446, 107: 1427, 117: 1340, 113: 1326, 109: 1312, 115: 1303, 111: 1293, 121: 1283, 119: 1201, 125: 1192, 123: 1185, 133: 1147, 131: 1143, 137: 1141, 129: 1138, 127: 1118, 135: 1056, 139: 1045, 143: 997, 141: 990, 149: 984, 145: 975, 147: 917, 157: 912, 151: 900, 153: 893, 155: 880, 159: 879, 167: 836, 161: 831, 169: 821, 165: 812, 175: 784, 163: 765, 171: 735, 177: 723, 181: 720, 173: 712, 179: 689, 189: 677, 187: 657, 183: 655, 185: 625, 191: 613, 195: 610, 203: 592, 193: 592, 199: 585, 207